# Step 2 — the labelled chip dataset

What `code/02_enmap_chips.py` produced: one 64×64 SWIR window per site, cut
from the EnMAP scene that covers it, saved with its mineral list.

    python code/02_enmap_chips.py --stage footprints   # ~1 h, once
    python code/02_enmap_chips.py --stage match
    python code/02_enmap_chips.py --stage chips --shard 0/4
    python code/02_enmap_chips.py --stage index

Read-only, as before.

In [ ]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "code"))
import common as C

# One fixed categorical order, used the same way in every chart and never
# cycled; a single hue carries magnitude. Validated for colour-vision
# deficiency, so series stay distinguishable.
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
SEQ = "Blues"
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#fcfcfb"
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.edgecolor": GRID, "axes.labelcolor": INK,
    "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "figure.figsize": (8, 4),
})

def report(name):
    """Load data/reports/<name>.json, or say which step writes it."""
    p = C.REPORTS / f"{name}.json"
    if not p.exists():
        raise SystemExit(f"{p.relative_to(REPO)} is missing — run the step that writes it.")
    return json.loads(p.read_text())

def need(path):
    p = Path(path)
    if not p.exists():
        raise SystemExit(f"{p} is missing — run the step that writes it first.")
    return p

def labelled_barh(ax, labels, values, color=CAT[0], fmt="{:,.0f}"):
    """Horizontal bars with the value written at the end of each one."""
    y = np.arange(len(labels))
    ax.barh(y, values, color=color, height=0.72)
    ax.set_yticks(y, labels)
    ax.invert_yaxis()
    ax.xaxis.set_visible(False)
    ax.grid(False)
    span = max(values) if len(values) else 1
    for yi, v in zip(y, values):
        ax.text(v + span * 0.01, yi, fmt.format(v), va="center", fontsize=9, color=MUTED)
    for s in ("left", "bottom"):
        ax.spines[s].set_visible(False)
    return ax


In [ ]:
match = report("02_match")
assign = pd.read_csv(need(C.SITE_SCENE_CSV), low_memory=False)
print(f"{match['sites_assigned']:,} sites assigned to {match['scenes_chosen']:,} scenes")
print(f"in-season: {100 * match['in_season_share']:.1f}%")
assign.head(3)

## From sites to scenes

The cloud filter is applied to the *scene* here, which is coarse — a 30 km
scene at 50% cloud can still be clear over a 1.9 km patch. The chip stage
measures cloud over the patch itself, which is what actually matters.

In [ ]:
rows = [("Sites from step 1", match["sites_in"]),
        ("EnMAP scenes in the catalogue", match["scenes_harvested"]),
        (f"Usable (cloud <= {match['cloud_max']}%)", match["scenes_usable"]),
        ("Sites with >= 1 usable scene", match["sites_covered"]),
        ("Sites with an in-season option", match["sites_with_in_season"]),
        ("Sites assigned a scene", match["sites_assigned"]),
        ("Scenes to read", match["scenes_chosen"])]
f = pd.DataFrame(rows, columns=["stage", "count"])
display(f.style.format({"count": "{:,}"}).hide(axis="index"))

fig, ax = plt.subplots(figsize=(8, 3))
labelled_barh(ax, f.stage, f["count"])
ax.set_xscale("log")
ax.set_title("Coverage funnel (log scale)", loc="left", pad=12)
plt.tight_layout()

## Which season the chips come from

Vegetation is the enemy of a SWIR mineral signal, so scenes are preferred from
the local dry / low-vegetation window — June–September north of the equator,
December–March south of it, unconstrained in the tropics where it barely
matters. Months are ordered, so one hue, light to dark.

In [ ]:
mh = pd.Series(match["month_histogram"]).sort_index()
mh.index = mh.index.astype(int)
names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
fig, ax = plt.subplots(figsize=(8, 3.5))
cols = plt.get_cmap(SEQ)(0.35 + 0.55 * mh.values / mh.values.max())
ax.bar([names[i - 1] for i in mh.index], mh.values, color=cols)
ax.set(ylabel="sites")
ax.set_title(f"Acquisition month of the assigned scene "
             f"({100 * match['in_season_share']:.0f}% in-season)", loc="left", pad=12)
plt.tight_layout()

## How the sites spread over scenes

Very skewed: a few scenes carry hundreds of sites, most carry one. That skew is
why scene choice is a greedy cover rather than a per-site decision — a per-site
choice would multiply the number of scenes to read without adding chips.

In [ ]:
pri = pd.read_csv(need(C.SCENE_PRIORITY_CSV))
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4))
axes[0].hist(pri.n_sites.clip(upper=50), bins=np.arange(0.5, 52), color=CAT[0])
axes[0].set(xlabel="sites in one scene (clipped at 50)", ylabel="scenes", yscale="log")
axes[1].plot(pri["rank"], 100 * pri.cumulative_sites / pri.n_sites.sum(),
             color=CAT[0], lw=2)
axes[1].set(xlabel="scenes read, biggest first", ylabel="% of sites covered")
for q in (50, 90):
    k = int((100 * pri.cumulative_sites / pri.n_sites.sum() >= q).idxmax()) + 1
    axes[1].axhline(q, color=GRID, lw=1)
    axes[1].annotate(f"{q}% at {k:,} scenes", (k, q), xytext=(6, -12),
                     textcoords="offset points", fontsize=9, color=MUTED)
plt.tight_layout()

## Build progress

Read straight off the shard manifests, so this is the honest state of the run.

In [ ]:
mans = sorted(C.CHIPS.glob("manifest*.csv"))
if not mans:
    print("no manifests yet — the chips stage has not run")
else:
    m = pd.concat([pd.read_csv(p).assign(manifest=p.name) for p in mans],
                  ignore_index=True)
    ok = m[m.status == "ok"]
    tot = int(ok.chips.sum() + ok.edge.sum() + ok.dropped.sum())
    summary = pd.DataFrame([
        ("scenes attempted", len(m)),
        ("scenes ok", len(ok)),
        ("scenes errored", int((m.status == "error").sum())),
        ("chips written", int(ok.chips.sum())),
        ("skipped: window off the scene edge", int(ok.edge.sum())),
        ("skipped: too cloudy or too masked", int(ok.dropped.sum())),
        ("survival rate", f"{100 * ok.chips.sum() / max(1, tot):.1f}%"),
    ], columns=["", "value"])
    display(summary.style.hide(axis="index"))
    if ok.seconds.sum() > 0:
        rate = ok.chips.sum() / (ok.seconds.sum() / 60)
        left = match["sites_assigned"] - ok.chips.sum()
        print(f"{rate:.0f} chips/min per worker; {left:,} chips left "
              f"≈ {left / max(1e-6, rate) / 60:.0f} worker-hours")
    if (m.status == "error").any():
        display(m[m.status == "error"].error.value_counts().head(10))

## The dataset as built

`data/chips/index.csv` is the file to load for training: one row per chip with
its path, label, cloud and valid fraction.

In [ ]:
ds = report("02_dataset")
idx = pd.read_csv(need(C.CHIP_INDEX_CSV), low_memory=False)
print(f"{ds['chips']:,} chips · {ds['gb']:.1f} GB · {ds['mb_per_chip']:.2f} MB each "
      f"· {ds['bands']} bands · {ds['chip_px']}×{ds['chip_px']} px")
if ds["duplicate_site_ids"]:
    print(f"WARNING {ds['duplicate_site_ids']:,} site ids appear twice — a site "
          f"was cut from more than one scene")
idx.head(3)

## Cloud and valid fraction

Two separate quantities, two charts. `valid` is the fraction of pixels not
flagged by the quality layers folded into the mask; all five layers are stored
raw in every chip, so this decision can be changed later without rebuilding.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4))
axes[0].hist(100 * idx.valid_frac, bins=np.linspace(0, 100, 41), color=CAT[0])
axes[0].set(xlabel="valid pixels (%)", ylabel="chips", yscale="log")
axes[0].set_title(f"mean {100 * idx.valid_frac.mean():.1f}%, "
                  f"{int((idx.valid_frac == 0).sum()):,} fully masked",
                  loc="left", fontsize=10, pad=10)
axes[1].hist(100 * idx.patch_cloud, bins=np.linspace(0, 100, 41), color=CAT[1])
axes[1].set(xlabel="cloud over the patch (%)", ylabel="chips", yscale="log")
axes[1].set_title(f"{int((idx.patch_cloud == 0).sum()):,} chips completely "
                  f"cloud-free", loc="left", fontsize=10, pad=10)
plt.tight_layout()

## A chip, actually looked at

A false-colour SWIR composite, the valid mask, the five quality layers, and the mean spectrum. If something is wrong with the build, it shows up here first.

In [ ]:
paths = sorted(C.CHIPS.glob("*/*.npz"))
if not paths:
    raise SystemExit("no chips yet")
d = np.load(paths[0], allow_pickle=False)
X, valid, wl = d["X"], d["valid"], d["wavelength"]
refl = np.where(valid[None] == 1, X * float(d["scale_factor"]), np.nan)

def band_at(nm):
    return int(np.argmin(np.abs(wl - nm)))

rgb = np.dstack([refl[band_at(nm)] for nm in (2200, 1650, 1050)])
lo, hi = np.nanpercentile(rgb, [2, 98])
rgb_s = np.clip((rgb - lo) / max(1e-9, hi - lo), 0, 1)

fig = plt.figure(figsize=(10, 6))
gs = fig.add_gridspec(2, 4, height_ratios=[1.2, 1])
ax = fig.add_subplot(gs[0, 0]); ax.imshow(np.nan_to_num(rgb_s)); ax.set_title("SWIR 2200/1650/1050 nm", fontsize=9)
ax2 = fig.add_subplot(gs[0, 1]); ax2.imshow(valid, cmap="gray", vmin=0, vmax=1); ax2.set_title(f"valid {valid.mean():.0%}", fontsize=9)
for a in (ax, ax2):
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
qn = [str(x) for x in d["qmask_names"]] if "qmask_names" in d else []
for i, name in enumerate(qn[:2]):
    a = fig.add_subplot(gs[0, 2 + i])
    a.imshow(d["qmask"][i], cmap=SEQ); a.set_title(f"{name} {d['patch_frac'][i]:.0%}", fontsize=9)
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
axs = fig.add_subplot(gs[1, :])
mean = np.nanmean(refl.reshape(len(wl), -1), axis=1)
axs.plot(wl, mean, color=CAT[0], lw=2)
axs.set(xlabel="wavelength (nm)", ylabel="reflectance")
axs.set_title(f"mean spectrum · site {str(d['site_id'])} · "
              f"{str(d['minerals'])[:70]}", loc="left", fontsize=9, pad=8)
plt.tight_layout()
print("gap at 1330–1450 nm is the water-vapour band, dropped on purpose")

## The label vocabulary over what was actually built

Step 1 counted minerals over every site; this counts them over the chips that exist, which is the vocabulary to train on.

In [ ]:
built = pd.read_csv(need(C.LABELS / "mineral_counts_built.csv"))
top = built.head(30)
fig, axes = plt.subplots(1, 2, figsize=(11, 7.5),
                         gridspec_kw={"width_ratios": [1.4, 1]})
labelled_barh(axes[0], top.mineral_name, top.n_chips)
axes[0].set_title(f"Chips per mineral (top 30 of {len(built):,}; "
                  f"{int(built.in_vocab.sum()):,} in vocabulary)", loc="left", pad=12)
axes[1].hist(idx.n_minerals.clip(upper=30), bins=np.arange(0.5, 32), color=CAT[0])
axes[1].set(xlabel="labels per chip (clipped at 30)", ylabel="chips")
axes[1].set_title("Multi-label, not single-label", loc="left", pad=12)
plt.tight_layout()

## Where the chips are

Coloured by acquisition month, so the seasonal hemisphere split is visible: northern sites cluster in summer months, southern ones in the opposite half of the year.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
s = ax.scatter(idx.lon, idx.lat, c=idx.month, cmap="twilight", s=1.2,
               alpha=0.6, linewidths=0, vmin=1, vmax=12)
ax.set(xlim=(-180, 180), ylim=(-90, 90), xlabel="longitude", ylabel="latitude")
ax.set_aspect("equal")
cb = fig.colorbar(s, ax=ax, shrink=0.8, ticks=range(1, 13))
cb.set_label("month")
ax.set_title(f"{len(idx):,} chips", loc="left", pad=12)
plt.tight_layout()